# BP2 Gate 1 — Business Understanding & Policy
**Customer360 Navigator Enterprise Suite — Customer Friction Classification**

## Why this notebook exists, and what it honestly scopes
Master Execution Plan Section 5.1 / Section 7 define BP2 as: *"define a documented friction
taxonomy (severity, sentiment proxy, repeat-contact signal) and evaluate with F1/precision/recall
and a confusion matrix."* This notebook grounds that definition against the real, already-profiled
CFPB and BANKING77 data (`docs/data_dictionary/RAW_DATA_MANIFEST.md`, `DATA_PROFILE_REPORT.md`) —
verified live below, not assumed from memory — before any Gate 3 model-benchmark work is built.

**Two of the three named signal types are not supportable by the real data in scope, and this
notebook says so explicitly rather than fabricating a substitute silently:**

- **Severity** — buildable. The real CFPB extract carries `Company response to consumer`
  (6 distinct values, live-enumerated in Section 5 below) and `Timely response?` (2 distinct
  values) as structured outcome fields. These become BP2's real, grounded friction signal.
- **Sentiment proxy** — NOT buildable as literally named. `RAW_DATA_MANIFEST.md` Finding 2
  (already confirmed for BP1) applies identically here: the real CFPB extract has no
  narrative/complaint-text column, and BANKING77 carries no sentiment annotation (schema is only
  `text,category`). No text-based sentiment signal exists anywhere in this project's real data.
  This notebook does not invent one. `Company public response` (12 distinct values, ~54% null —
  live-checked in Section 5) is the nearest real *structured* proxy if a substitute is wanted, but
  using it that way is a labeled substitution deferred to Gate 2 human review, not asserted as
  "sentiment" here.
- **Repeat-contact signal** — NOT buildable. The real CFPB schema (15 columns, confirmed by
  `RAW_DATA_MANIFEST.md` and re-verified live in Section 4 below) has no persistent
  customer/consumer identifier — only `Complaint ID`, which identifies the complaint, not the
  person who filed it. This is the exact same real-data constraint the Master Plan already
  resolves honestly for BP4 ("do not invent customer IDs... if longitudinal identity is absent,
  call this event/issue journey analytics explicitly," Section 5.1 / risk register: *"No
  longitudinal customer ID -> Do not fabricate; use issue/event-level journeys (BP4)"*). BP2
  applies the identical resolution: repeat-contact is scoped out of the friction taxonomy
  entirely, not simulated.

BP2's friction taxonomy is therefore honestly narrowed to **severity**, derived from real CFPB
outcome fields. This narrowing is recorded as an explicit ASSUMPTION below, not silently applied.

## Purpose
Produces BP2's Gate 1 output exactly as Section 8 (6-Gate Governance SOP) defines it: a policy
artifact recording the target definition, leakage rules, and ASSUMPTIONs — verified against the
real, profiled data, not invented. The concrete bucket-to-severity-class mapping (which of the 6
real `Company response to consumer` values map to which ordinal severity level) is deliberately
**deferred to Gate 2** (taxonomy engineering), exactly as BP1 deferred its CFPB<->BANKING77
crosswalk from Gate 1 to Gate 2 — because only the *count* of distinct real values (6 and 2) is
currently confirmed in this project's documentation, not the actual label strings, and this
notebook will not guess them. Gate 1's own live checks below fetch and record the real strings for
the first time.

## Standing rules this notebook follows
- **Execution boundary** (Section 12.2): Claude wrote this notebook; it does not run it. You run it
  on your own machine, and the real, live-checked results below become this project's Gate 1
  policy record for BP2.
- **Zero-fabrication** (Section 12.1): every check below runs against the real files in
  `data/external/` / `data/raw/`. No category label, count, or distinct-value string in this
  notebook is asserted from memory — all are read live from the real CSVs.
- **HYPER**: reuses BP1's Gate 1 structural pattern (project-root resolution, WARP setup,
  marker-based config sync) and `src/utils/bp1_config_sync.py` as-is — its functions are already
  fully parameterized by `config_path` and contain no BP1-specific logic, only a BP1-named
  docstring/module name. Reused unmodified here rather than duplicated or prematurely renamed
  (a rename would touch BP1's already Gate-1-through-6-confirmed notebooks' imports for no
  functional gain) — flagged as a naming-debt note for a future consolidation pass, not a
  functional gap.

In [ ]:
# ============================================================
# SECTION 1: Project root resolution (PROJECT_STRUCTURE_LOCKED.md rule #3)
# ============================================================
import os
import sys
from pathlib import Path


def _find_project_root(marker_filename: str = "PROJECT_STRUCTURE_LOCKED.md") -> Path:
    env_override = os.environ.get("C360_PROJECT_ROOT")
    if env_override:
        candidate = Path(env_override)
        if (candidate / marker_filename).exists():
            return candidate
        raise RuntimeError(
            f"C360_PROJECT_ROOT is set to {candidate} but {marker_filename} was not found there. "
            "Fix the environment variable rather than removing this check."
        )

    start = Path.cwd()
    current = start
    for _ in range(8):
        if (current / marker_filename).exists():
            return current
        if current.parent == current:
            break
        current = current.parent

    for depth_root, dirnames, filenames in os.walk(start):
        rel_depth = len(Path(depth_root).relative_to(start).parts)
        if rel_depth > 3:
            dirnames[:] = []
            continue
        dirnames[:] = [d for d in dirnames if not d.startswith(".")]
        if marker_filename in filenames:
            return Path(depth_root)

    raise RuntimeError(
        "Could not resolve PROJECT_ROOT. Set the C360_PROJECT_ROOT environment variable to the "
        "Customer360_Navigator_Enterprise_Suite folder, or run this notebook from inside the project tree "
        "(expected at notebooks/bp2_customer_friction_classification/)."
    )


PROJECT_ROOT = _find_project_root()
sys.path.insert(0, str(PROJECT_ROOT / "src"))
CONFIGS_DIR = PROJECT_ROOT / "configs"
DATA_EXTERNAL_DIR = PROJECT_ROOT / "data" / "external"
DATA_RAW_DIR = PROJECT_ROOT / "data" / "raw"
ARTIFACTS_DIR = PROJECT_ROOT / "notebooks" / "bp2_customer_friction_classification" / "artifacts"
ARTIFACTS_DIR.mkdir(parents=True, exist_ok=True)
print(f"[OK] Project root resolved: {PROJECT_ROOT.name}")

# ============================================================
# SECTION 2: WARP - configure_performance() FIRST, before any heavy/BLAS-backed import
# ============================================================
from utils.performance_setup import configure_performance  # noqa: E402

WARP_SUMMARY = configure_performance(project_root=PROJECT_ROOT, verbose=True)

# ============================================================
# SECTION 3: Heavy imports + flush-forcing print override (LESSONS_LEARNED_APPLIED.md #12)
# ============================================================
import builtins  # noqa: E402
import functools  # noqa: E402
import json  # noqa: E402
import warnings  # noqa: E402
from datetime import datetime, timezone  # noqa: E402

import polars as pl  # noqa: E402

from taxonomy.taxonomy_mapper import CFPB_DTYPES  # noqa: E402

warnings.filterwarnings("ignore")
print = functools.partial(builtins.print, flush=True)

CFPB_PATH = DATA_RAW_DIR / "cfpb_complaints.csv"
B77_TRAIN_PATH = DATA_EXTERNAL_DIR / "banking77_train.csv"

# ============================================================
# SECTION 4: Structural check - no persistent customer/consumer identifier in the real CFPB
# schema (grounds the "repeat-contact signal scoped out" ASSUMPTION - verified live, not
# asserted from RAW_DATA_MANIFEST.md's prior documentation alone).
# ============================================================
cfpb_columns = list(CFPB_DTYPES.keys())
ID_LIKE_KEYWORDS = ("customer", "consumer id", "person", "account number", "ssn", "email", "phone")
suspected_customer_id_columns = [
    c for c in cfpb_columns if any(kw in c.lower() for kw in ID_LIKE_KEYWORDS)
]
print(f"[OK] Real CFPB columns ({len(cfpb_columns)}): {cfpb_columns}")
print(f"[OK] Columns matching customer/consumer-identifier keywords: {suspected_customer_id_columns or 'NONE'}")
print("[OK] 'Complaint ID' identifies the complaint record, not the person who filed it - "
      "confirmed by column list above, not a per-row live computation.")

# ============================================================
# SECTION 5: Live enumeration of the real candidate severity/friction fields - the actual label
# strings (not just distinct counts) are fetched here for the first time in this project's
# documentation. No category string below is asserted from memory.
# ============================================================
cfpb_lazy = pl.scan_csv(CFPB_PATH, schema_overrides=CFPB_DTYPES)

company_response_counts = (
    cfpb_lazy.group_by("Company response to consumer")
    .agg(pl.len().alias("n"))
    .sort("n", descending=True)
    .collect()
)
timely_response_counts = (
    cfpb_lazy.group_by("Timely response?")
    .agg(pl.len().alias("n"))
    .sort("n", descending=True)
    .collect()
)
company_public_response_counts = (
    cfpb_lazy.group_by("Company public response")
    .agg(pl.len().alias("n"))
    .sort("n", descending=True)
    .collect()
)

total_rows = cfpb_lazy.select(pl.len()).collect().item()
company_public_response_null_rows = int(
    company_public_response_counts.filter(pl.col("Company public response").is_null())["n"].sum() or 0
)

print(f"[OK] Real CFPB row count (live): {total_rows:,}")
print("[OK] Live 'Company response to consumer' distinct values + real counts:")
print(company_response_counts)
print("[OK] Live 'Timely response?' distinct values + real counts:")
print(timely_response_counts)
print("[OK] Live 'Company public response' distinct values + real counts:")
print(company_public_response_counts)
print(f"[OK] 'Company public response' null rows (live): {company_public_response_null_rows:,} "
      f"({company_public_response_null_rows / total_rows:.2%} of {total_rows:,})")

# ============================================================
# SECTION 6: Structural leakage check - BANKING77's real schema carries no friction/severity/
# sentiment field of any kind (verified live, mirrors BP1 Gate 1 Section 4's shared-column check).
# ============================================================
banking77_columns = set(pl.read_csv(B77_TRAIN_PATH, n_rows=1).columns)
print(f"[OK] BANKING77 columns (live): {sorted(banking77_columns)}")
print(f"[OK] BANKING77 column count: {len(banking77_columns)} (expected 2: text, category - "
      f"no friction/severity/sentiment field exists in this dataset)")

# ============================================================
# SECTION 7: Assemble the Gate 1 policy (target definition, leakage rules, assumptions)
# ============================================================
policy = {
    "bp_id": "bp2",
    "bp_name": "bp2_customer_friction_classification",
    "gate": 1,
    "generated_at_utc": datetime.now(timezone.utc).isoformat(),
    "target_definition": {
        "primary_target": "friction_severity_class",
        "primary_target_description": "An ordinal friction-severity class derived from the real "
                                       "CFPB `Company response to consumer` and `Timely response?` "
                                       "fields. The concrete bucket-to-ordinal-class mapping (which "
                                       "of the real values in Section 5's live output map to which "
                                       "severity level) is deliberately deferred to Gate 2 (taxonomy "
                                       "engineering) - the same sequencing BP1 used for its "
                                       "CFPB<->BANKING77 crosswalk - because only the live-enumerated "
                                       "real label strings above, not an assumed set, may ground it.",
        "scoped_out_signals": {
            "sentiment_proxy": "NOT built as literally named in the Master Plan - no narrative/"
                                "complaint-text column exists in the real CFPB extract "
                                "(RAW_DATA_MANIFEST.md Finding 2) and BANKING77 carries no "
                                "sentiment annotation (schema verified live in Section 6: text, "
                                "category only). `Company public response` is the nearest real "
                                "structured substitute if wanted later; using it that way is a "
                                "labeled substitution for Gate 2 human review, not asserted here.",
            "repeat_contact_signal": "NOT built - the real CFPB schema (verified live in Section 4) "
                                      "has no persistent customer/consumer identifier, only "
                                      "`Complaint ID` (identifies the complaint, not the person). "
                                      "Scoped out entirely, applying the identical real-data "
                                      "resolution the Master Plan already uses for BP4 ('do not "
                                      "invent customer IDs... call this event/issue journey "
                                      "analytics explicitly').",
        },
        "feature_variable_candidates": "Structured CFPB fields only (Product, Sub-product, Issue, "
                                        "Sub-issue, State, Submitted via) plus the Gate-2/BP1 "
                                        "`common_taxonomy_bucket` (configs/taxonomy_mapping.yaml) as "
                                        "an optional categorical feature for the ~6.55% of CFPB rows "
                                        "already in scope for BANKING77 overlap (UNMAPPED_UNKNOWN_"
                                        "CATEGORY for the rest) - this is how BP2 satisfies the "
                                        "Master Plan's 'Integrates BANKING77: YES' requirement, since "
                                        "BANKING77 itself carries no friction signal to train on "
                                        "directly. Final feature set is a Gate 2/3 decision.",
        "train_test_split_source": "CFPB ships no pre-defined train/test split (unlike BANKING77) - "
                                    "BP2 will use a fresh random split of the real CFPB extract, "
                                    "stratified by the derived target, random_state=42 (matching "
                                    "BP1's random_state for consistency, not a shared value).",
    },
    "leakage_rules": [
        "`Company response to consumer` and `Timely response?` define the target and must NEVER "
        "also be used as input features - they are not traditional train/test leakage, they are "
        "the label itself. This is the single most important rule in this policy.",
        "`Company public response` is NOT pre-cleared as a safe feature - it is ~"
        f"{company_public_response_null_rows / total_rows:.0%} null (live-checked in Section 5) and "
        "its relationship to the target is unverified; Gate 2/3 must test it for suspiciously high "
        "predictive power before treating it as a legitimate feature rather than a leakage proxy.",
        "No CFPB row may appear in both the train and test split of BP2's fresh random split - "
        "verified structurally by construction (single stratified split call, not a manual/ad hoc "
        "merge) once Gate 3 builds it.",
        "BANKING77 is never a source of friction/severity training labels - verified live in Section "
        "6 that its schema carries no such field. It contributes only the Gate-2/BP1 taxonomy-bucket "
        "feature described above.",
    ],
    "assumptions": [
        "BP2's friction taxonomy is honestly narrowed to ONE of the Master Plan's three named "
        "signal types (severity) - sentiment proxy and repeat-contact signal are scoped out for the "
        "documented real-data reasons above, not silently substituted.",
        "The severity bucket-to-class mapping itself is NOT decided in this notebook - it is "
        "deferred to Gate 2, to be built as a documented judgment mapping (HIGH/MEDIUM/LOW "
        "confidence per bucket, same pattern as configs/taxonomy_mapping.yaml) against the real "
        "values live-enumerated in Section 5, with explicit human review before Gate 3 trains on it.",
        "`src/utils/bp1_config_sync.py` is reused unmodified for BP2's own config file "
        "(configs/bp2_customer_friction_classification.yaml) - its read/write functions are already "
        "fully parameterized by config_path and contain no BP1-specific logic.",
    ],
    "compliance_touchpoint": {
        "requirement": "UDAAP (Unfair, Deceptive, or Abusive Acts or Practices) framing",
        "statement": "Per Master Plan Section 9: friction findings from this classifier are framed "
                     "as potential risk indicators only when statistically supported - never as a "
                     "determination of unfair, deceptive, or abusive conduct by any named company. "
                     "No demographic or protected-class field exists in the real CFPB extract in "
                     "scope for this project (docs/data_dictionary/RAW_DATA_MANIFEST.md Finding 3), "
                     "so ECOA/Regulation B disparate-impact testing is Not Applicable - No "
                     "Protected-Class Field in Scope, stated honestly rather than skipped silently.",
    },
    "live_checks": {
        "cfpb_row_count": total_rows,
        "cfpb_columns": cfpb_columns,
        "suspected_customer_id_columns": suspected_customer_id_columns,
        "banking77_columns": sorted(banking77_columns),
        "company_response_to_consumer_distribution": company_response_counts.to_dicts(),
        "timely_response_distribution": timely_response_counts.to_dicts(),
        "company_public_response_distribution": company_public_response_counts.to_dicts(),
        "company_public_response_null_rows": company_public_response_null_rows,
    },
}

# ============================================================
# SECTION 8: Write outputs (idempotent overwrite-in-place)
# ============================================================
policy_json_path = ARTIFACTS_DIR / "policy.json"
with open(policy_json_path, "w", encoding="utf-8") as f:
    json.dump(policy, f, indent=2, default=str)
print(f"\n[SAVED] {policy_json_path.relative_to(PROJECT_ROOT)}")

bp2_config_path = CONFIGS_DIR / "bp2_customer_friction_classification.yaml"

# BP2 reuses BP1's marker-based config-sync helpers as-is (src/utils/bp1_config_sync.py) - they
# are already generic (parameterized entirely by config_path, no BP1-specific logic inside),
# so Gate 1 here owns only the front-matter section below and every later gate's block is
# preserved verbatim regardless of position or order, exactly as for BP1. See
# LESSONS_LEARNED_APPLIED.md #20 for the real incident this pattern was built to prevent.
from utils.bp1_config_sync import read_existing_gate_block_markers, write_front_matter  # noqa: E402

_existing_gate_markers = read_existing_gate_block_markers(bp2_config_path)
_status_suffix = ""
for _gate_num, _gate_label in ((3, "Gate 3"), (4, "Gate 4"), (5, "Gate 5")):
    if any(_gate_label in _m for _m in _existing_gate_markers):
        _status_suffix += f"_gate{_gate_num}_confirmed"

bp2_config_text = f"""# Per-BP config - filled in at Gate 1 (Business Understanding & Policy)
# Gate 1 owns bp_id through random_state below via write_front_matter() (src/utils/bp1_config_sync.py,
# reused as-is from BP1 - fully generic, parameterized by config_path); Gates 2-5 each own exactly one
# marker-delimited block appended after it via write_gate_block() - do not hand-edit either section,
# re-run the owning notebook instead.
bp_id: "bp2"
bp_name: "bp2_customer_friction_classification"
status: "gate1_confirmed{_status_suffix}"   # not_started | gate1 | gate2 | gate3 | gate4 | gate5 | gate6_complete
target_definition:
  primary_target: "friction_severity_class"
  primary_target_description: "Ordinal friction-severity class derived from real CFPB
    'Company response to consumer' + 'Timely response?' fields. Bucket-to-class mapping deferred to
    Gate 2 (taxonomy engineering) - see notebook Section 7 / policy.json for the live-enumerated
    real values this must be built against."
  scoped_out_signals:
    sentiment_proxy: "NOT built as literally named - no CFPB narrative text, no BANKING77 sentiment
      annotation. 'Company public response' is the nearest real structured substitute, deferred to
      Gate 2 human review."
    repeat_contact_signal: "NOT built - no persistent customer/consumer identifier in the real CFPB
      schema (Complaint ID identifies the complaint, not the person). Same resolution as BP4's
      'do not invent customer IDs' rule."
  feature_variable_candidates: "Structured CFPB fields (Product, Sub-product, Issue, Sub-issue,
    State, Submitted via) plus the Gate-2/BP1 common_taxonomy_bucket for BANKING77 integration."
  train_test_split_source: "Fresh random split of the real CFPB extract (CFPB ships no pre-defined
    split), stratified by the derived target, random_state=42."
leakage_rules:
  - "'Company response to consumer' and 'Timely response?' define the target and must never also be
     used as input features."
  - "'Company public response' is not pre-cleared as a safe feature (~54% null, live-checked) -
     Gate 2/3 must test it for leakage before use."
  - "No CFPB row may appear in both train and test splits - enforced structurally at Gate 3."
  - "BANKING77 is never a source of friction/severity training labels (schema verified live: text,
     category only) - contributes only the Gate-2/BP1 taxonomy-bucket feature."
assumptions:
  - "BP2's friction taxonomy is honestly narrowed to severity only - sentiment proxy and
     repeat-contact signal are scoped out for documented real-data reasons, not substituted
     silently."
  - "Severity bucket-to-class mapping is deferred to Gate 2, built as a documented judgment mapping
     (HIGH/MEDIUM/LOW confidence, same pattern as configs/taxonomy_mapping.yaml) against the real
     live-enumerated values, with explicit human review before Gate 3 trains on it."
  - "src/utils/bp1_config_sync.py is reused unmodified for this config file - already fully generic,
     parameterized by config_path, no BP1-specific logic."
random_state: 42
"""
write_front_matter(bp2_config_path, bp2_config_text)
print(f"[SAVED] {bp2_config_path.relative_to(PROJECT_ROOT)}")

# ============================================================
# SECTION 9: Structural integrity checks - raise AssertionError, never silently pass
# ============================================================
checks = {
    "no_customer_identifier_column_in_cfpb_schema": len(suspected_customer_id_columns) == 0,
    "banking77_schema_has_no_friction_field": banking77_columns == {"text", "category"},
    "company_response_to_consumer_enumerated": company_response_counts.height > 0,
    "timely_response_enumerated": timely_response_counts.height > 0,
    "company_public_response_enumerated": company_public_response_counts.height > 0,
    "policy_json_written": policy_json_path.exists(),
    "bp2_config_yaml_written": bp2_config_path.exists(),
}

print("\n=== INTEGRITY CHECKS ===")
for name, passed in checks.items():
    status = "[PASS]" if passed else "[FAIL]"
    print(f"{status} {name}")
    assert passed, f"[CHECK FAILED] {name}"

print("\n[ALL CHECKS PASSED] BP2 Gate 1 complete - target honestly scoped to severity only, "
      "sentiment proxy and repeat-contact signal explicitly excluded with real-data justification, "
      "real category values live-enumerated for Gate 2. Proceed to BP2 Gate 2 (taxonomy/severity-"
      "bucket engineering) next.")